In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import geopandas as gpd
from shapely.geometry import Point, LineString, shape
import json

def load_and_clean_data():
    # some columns have bad lines, so need to skip them to avoid errors and csv data columns are separated by semicolons
    df_streets = pd.read_csv('datasets/public-streets.csv', delimiter = ';', on_bad_lines='skip') 
    df_streets.drop('STREETUSE', axis=1, inplace=True)

    df_pavement = pd.read_csv('datasets/pavement-conditions.csv', delimiter = ';', on_bad_lines='skip') 
    df_pavement['Year'] = pd.to_numeric(df_pavement['Year'], errors='coerce') # coerce errors to NaN
    df_pavement['length_(m)'] = pd.to_numeric(df_pavement['length_(m)'], errors='coerce')
    df_pavement = df_pavement.dropna()

    # One Hot Encoding the ratings
    one_hot = pd.get_dummies(df_pavement['PCI Rating'])
    df_pavement = df_pavement.join(one_hot)
    df_pavement.drop('PCI Rating', axis=1, inplace=True)  # Drop the original PCI Rating column

    df_streetlights = pd.read_csv('datasets/street-lighting-poles.csv', delimiter = ';', on_bad_lines='skip') 
    df_streetlights = df_streetlights.dropna()
    df_streetlights.drop(columns=['Geo Local Area', 'BLOCK_NUMBER', 'NODE_NUMBER'], axis = 1, inplace = True) 

    df_construction = pd.read_csv('datasets/roads-under-construction.csv', delimiter = ';', on_bad_lines='skip') 
    # df_construction.isna().sum() #removing na values results in an empty dataframe
    df_construction.drop(['PROJECT', 'URL_LINK', 'STREET'], axis=1, inplace=True) # remove columns that are not needed
    df_construction = df_construction.rename(columns={'LOCATION': 'Street'})

    df_curb_ramps = pd.read_csv('datasets/curb-ramp-priorities.csv', delimiter = ',', on_bad_lines='skip')
    df_curb_ramps = df_curb_ramps.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

    print(f"Loaded datasets:\nAll Streets: {df_streets.shape} \nPavement Conditions: {df_pavement.shape} "
          f"\nStreetlights: {df_streetlights.shape} \nConstruction: {df_construction.shape} "
          f"\nCurb Ramp Priorities: {df_curb_ramps.shape}")
    
    return df_streets, df_pavement, df_streetlights, df_construction, df_curb_ramps

In [3]:
df_streets, df_pavement, df_streetlights, df_construction, df_curb_ramps = load_and_clean_data()
df_construction.columns

Loaded datasets:
All Streets: (17063, 3) 
Pavement Conditions: (5241, 13) 
Streetlights: (57514, 2) 
Construction: (32, 4) 
Curb Ramp Priorities: (149, 3)


Index(['Geom', 'Street', 'COMP_DATE', 'geo_point_2d'], dtype='object')

# Getting Latitude/Longitude and Geometry

In [4]:
def extract_lat_lon(df):
    df[['latitude', 'longitude']] = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df.drop('geo_point_2d', axis=1, inplace=True)
    return df

# Convert Geom to geometry
def convert_to_geometry(df):
    df_copy = df.copy()
    
    if 'Geom' in df_copy.columns:
        df_copy['geometry'] = df_copy['Geom'].apply(lambda x: shape(json.loads(x)))
        df_copy.drop('Geom', axis=1, inplace=True)
    elif 'latitude' in df_copy.columns and 'longitude' in df_copy.columns: #for curb ramps dataset
        # lambda row: Point(row['longitude'], row['latitude']) creates a Point object for each row
        # using the longitude and latitude columns
        df_copy['geometry'] = df_copy.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
    else:
        raise ValueError("No 'Geom' or 'latitude' and 'longitude' columns found in dataframe to convert to geometry.")
    
    gdf = gpd.GeoDataFrame(df_copy, geometry='geometry', crs='EPSG:4326') #crs epsg:4326 is WGS 84 which is a common coordinate reference system for geographic data

    gdf = gdf.to_crs('EPSG:26910') #convert to crs EPSG:26910 that is used for UTM Zone 10N (expressed in meters) that's used by vancouver open data

    return gdf

df_streets = extract_lat_lon(df_streets)
df_pavement = extract_lat_lon(df_pavement)
df_streetlights = extract_lat_lon(df_streetlights)
df_construction = extract_lat_lon(df_construction)

gdf_streets = convert_to_geometry(df_streets)
gdf_pavement = convert_to_geometry(df_pavement)
gdf_streetlights = convert_to_geometry(df_streetlights)
gdf_construction = convert_to_geometry(df_construction)
gdf_curb_ramps = convert_to_geometry(df_curb_ramps)


In [5]:
print('\ngdf_streets columns: ', gdf_streets.columns, '\n', gdf_streets.head(2))
print('\ngdf_pavement columns: ', gdf_pavement.columns, '\n', gdf_pavement.head(2))
print('\ngdf_streetlights columns: ', gdf_streetlights.columns, '\n', gdf_streetlights.head(2))
print('\ngdf_construction columns: ', gdf_construction.columns, '\n', gdf_construction.head(2))
print('\ngdf_curb_ramps columns: ', gdf_curb_ramps.columns, '\n', gdf_curb_ramps.head(2))
print(gdf_streets.shape, gdf_pavement.shape, gdf_streetlights.shape, gdf_construction.shape, gdf_curb_ramps.shape)


gdf_streets columns:  Index(['HBLOCK', 'latitude', 'longitude', 'geometry'], dtype='object') 
            HBLOCK   latitude   longitude  \
0  3200 E 45TH AV  49.229275 -123.036582   
1  5800 RUPERT ST  49.230424 -123.043113   

                                            geometry  
0  LINESTRING (497324.29 5452944.313, 497348.96 5...  
1  LINESTRING (496861.279 5453097.122, 496861.008...  

gdf_pavement columns:  Index(['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)', 'FAIR',
       'GOOD', 'NO DATA', 'POOR', 'VERY GOOD', 'VERY POOR', 'latitude',
       'longitude', 'geometry'],
      dtype='object') 
    Year Road Name               From Street    To Street  length_(m)   FAIR  \
0  2021  41ST AVE              INVERNESS ST    KNIGHT ST          50  False   
1  2021   1ST AVE  EB STOP BAR AT RUPERT ST  BOUNDARY RD          23  False   

    GOOD  NO DATA   POOR  VERY GOOD  VERY POOR   latitude   longitude  \
0  False    False  False       True      False  49.232925 -123.0

# Merging Pavement Data to Base Street Data

In [6]:
def create_buffer(gdf, distance):
    # Create a small buffer around street geometries 
    gdf_buffered = gdf.copy()
    gdf_buffered['geometry'] = gdf_buffered.geometry.buffer(distance)
    return gdf_buffered

def merge_pavement_to_streets(gdf_streets, gdf_pavement, max_distance_meters=100):

    # sjoin_nearest will find the nearest pavement condition to each street segment by doing a Spatial join of those two GeoDataFrames based on the distance between their geometries
    # how='left' ensures that all streets are retained, even if they don't have a matching pavement condition
    # max_distance = 100 meters
    streets_pavement = gpd.sjoin_nearest(
        gdf_streets, 
        gdf_pavement[['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)', 
                     'FAIR', 'GOOD', 'NO DATA', 'POOR', 'VERY GOOD', 'VERY POOR', 'geometry']], 
        how='left',
        max_distance=max_distance_meters  # max distance for matching
    )
   
    streets_pavement = streets_pavement.drop(columns=['index_right'], errors='ignore')  # drop index_right column if it exists
    pavement_cols = ['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)', 
                    'FAIR', 'GOOD', 'NO DATA', 'POOR', 'VERY GOOD', 'VERY POOR']
    pavement_rename = []
    for col in pavement_cols:
        pavement_rename.append(f'pavement_{col}')
    streets_pavement = streets_pavement.rename(columns=dict(zip(pavement_cols, pavement_rename)))

    return streets_pavement

merged_pavement = merge_pavement_to_streets(gdf_streets, gdf_pavement) # using buffered streets results in more pavement conditions (66944) being matched to streets vs using original streets results in ~1 or a bit more pavement conditions (17090) being matched to streets 
print(f"merged_pavement shape after merge: {merged_pavement.shape}")
print(merged_pavement.head(5))

merged_pavement shape after merge: (17090, 15)
               HBLOCK   latitude   longitude  \
0      3200 E 45TH AV  49.229275 -123.036582   
1      5800 RUPERT ST  49.230424 -123.043113   
2  3800 MARGUERITE ST  49.251417 -123.143891   
3     1600 MARPOLE AV  49.255756 -123.141315   
4      1400 W 15TH AV  49.257967 -123.137180   

                                            geometry  pavement_Year  \
0  LINESTRING (497324.29 5452944.313, 497348.96 5...            NaN   
1  LINESTRING (496861.279 5453097.122, 496861.008...            NaN   
2  LINESTRING (489551.591 5455470.774, 489529.845...            NaN   
3  LINESTRING (489780.899 5455911.021, 489777.22 ...         2021.0   
4  LINESTRING (490081.879 5456140.711, 489954.518...         2021.0   

  pavement_Road Name            pavement_From Street       pavement_To Street  \
0                NaN                             NaN                      NaN   
1                NaN                             NaN                      N

# Merging Streetlights data to Base Streets Dataset

In [7]:
def merge_streetlights_to_streets(gdf_dataset, gdf_streetlights, buffer_distance_meters=50):
    streets_lights = gdf_dataset.copy()
    street_buffered = create_buffer(streets_lights, buffer_distance_meters)  # ~50m buffer to match streetlights within 50m of streets

    streets_lights['streetlight_count'] = 0
    streets_lights['avg_distance_to_streetlight_meters'] = np.nan


    for index, street_row in street_buffered.iterrows():
        lights_within_buffer = gdf_streetlights[gdf_streetlights.geometry.within(street_row.geometry)]
        if len(lights_within_buffer) > 0:
            streets_lights.at[index, 'streetlight_count'] = len(lights_within_buffer)
            street_geom = streets_lights.at[index, 'geometry']
            distances = [street_geom.distance(light_geom) for light_geom in lights_within_buffer.geometry]
            streets_lights.at[index, 'avg_distance_to_streetlight_meters'] = np.mean(distances)
    return streets_lights

merged_streetlights = merge_streetlights_to_streets(merged_pavement, gdf_streetlights)
print(f"merged_streetlights shape after merge: {merged_streetlights.shape}")
print(merged_streetlights.head(5))

merged_streetlights shape after merge: (17090, 17)
               HBLOCK   latitude   longitude  \
0      3200 E 45TH AV  49.229275 -123.036582   
1      5800 RUPERT ST  49.230424 -123.043113   
2  3800 MARGUERITE ST  49.251417 -123.143891   
3     1600 MARPOLE AV  49.255756 -123.141315   
4      1400 W 15TH AV  49.257967 -123.137180   

                                            geometry  pavement_Year  \
0  LINESTRING (497324.29 5452944.313, 497348.96 5...            NaN   
1  LINESTRING (496861.279 5453097.122, 496861.008...            NaN   
2  LINESTRING (489551.591 5455470.774, 489529.845...            NaN   
3  LINESTRING (489780.899 5455911.021, 489777.22 ...         2021.0   
4  LINESTRING (490081.879 5456140.711, 489954.518...         2021.0   

  pavement_Road Name            pavement_From Street       pavement_To Street  \
0                NaN                             NaN                      NaN   
1                NaN                             NaN                   

# Merging Curb Ramps Data to Base Streets Dataset

In [8]:
def merge_curb_ramps_to_streets(gdf_streets, gdf_curb_ramps, buffer_distance_meters=50):
    streets_curb_ramps = gdf_streets.copy()
    street_buffered = create_buffer(streets_curb_ramps, buffer_distance_meters)  # ~50m buffer to match curb ramps within 50m of streets

    streets_curb_ramps['curb_ramp_count'] = 0
    streets_curb_ramps['avg_distance_to_curb_ramp_meters'] = np.nan

    for index, street_row in street_buffered.iterrows():
        ramps_within_buffer = gdf_curb_ramps[gdf_curb_ramps.geometry.within(street_row.geometry)]
        if len(ramps_within_buffer) > 0:
            streets_curb_ramps.at[index, 'curb_ramp_count'] = len(ramps_within_buffer)
            street_geom = streets_curb_ramps.at[index, 'geometry']
            distances = [street_geom.distance(ramp_geom) for ramp_geom in ramps_within_buffer.geometry]
            streets_curb_ramps.at[index, 'avg_distance_to_curb_ramp_meters'] = np.mean(distances)
    return streets_curb_ramps

merged_curb_ramps = merge_curb_ramps_to_streets(merged_streetlights, gdf_curb_ramps)
print(f"merged_curb_ramps shape after merge: {merged_curb_ramps.shape}")
print(merged_curb_ramps.head(5))

merged_curb_ramps shape after merge: (17090, 19)
               HBLOCK   latitude   longitude  \
0      3200 E 45TH AV  49.229275 -123.036582   
1      5800 RUPERT ST  49.230424 -123.043113   
2  3800 MARGUERITE ST  49.251417 -123.143891   
3     1600 MARPOLE AV  49.255756 -123.141315   
4      1400 W 15TH AV  49.257967 -123.137180   

                                            geometry  pavement_Year  \
0  LINESTRING (497324.29 5452944.313, 497348.96 5...            NaN   
1  LINESTRING (496861.279 5453097.122, 496861.008...            NaN   
2  LINESTRING (489551.591 5455470.774, 489529.845...            NaN   
3  LINESTRING (489780.899 5455911.021, 489777.22 ...         2021.0   
4  LINESTRING (490081.879 5456140.711, 489954.518...         2021.0   

  pavement_Road Name            pavement_From Street       pavement_To Street  \
0                NaN                             NaN                      NaN   
1                NaN                             NaN                     

# Merge Construction Data to Base Streets Dataset

In [9]:
def merge_construction_to_streets(gdf_streets, gdf_construction, max_distance_meters=100):
    streets_construction = gdf_streets.copy()
    streets_construction['under_construction'] = False
    streets_construction['construction_distance'] = np.nan
    streets_construction['construction_location'] = None
    streets_construction['construction_completion'] = None

    for index, street_row in streets_construction.iterrows():
        distances = gdf_construction.geometry.distance(street_row.geometry)
        min_distance = distances.min()

        if min_distance <= max_distance_meters: # if the minimum distance to a construction site is within the specified max distance
            closest_construction = gdf_construction.loc[distances.idxmin()]
            streets_construction.at[index, 'under_construction'] = True
            streets_construction.at[index, 'construction_distance'] = min_distance
            streets_construction.at[index, 'construction_location'] = closest_construction['Street']
            streets_construction.at[index, 'construction_completion'] = closest_construction['COMP_DATE']

    return streets_construction

merged_construction = merge_construction_to_streets(merged_curb_ramps, gdf_construction)
print(f"merged_construction shape after merge: {merged_construction.shape}")
print(merged_construction.head(5))
print(merged_construction['under_construction'].value_counts())


merged_construction shape after merge: (17090, 23)
               HBLOCK   latitude   longitude  \
0      3200 E 45TH AV  49.229275 -123.036582   
1      5800 RUPERT ST  49.230424 -123.043113   
2  3800 MARGUERITE ST  49.251417 -123.143891   
3     1600 MARPOLE AV  49.255756 -123.141315   
4      1400 W 15TH AV  49.257967 -123.137180   

                                            geometry  pavement_Year  \
0  LINESTRING (497324.29 5452944.313, 497348.96 5...            NaN   
1  LINESTRING (496861.279 5453097.122, 496861.008...            NaN   
2  LINESTRING (489551.591 5455470.774, 489529.845...            NaN   
3  LINESTRING (489780.899 5455911.021, 489777.22 ...         2021.0   
4  LINESTRING (490081.879 5456140.711, 489954.518...         2021.0   

  pavement_Road Name            pavement_From Street       pavement_To Street  \
0                NaN                             NaN                      NaN   
1                NaN                             NaN                   

# Export final dataset

In [14]:
final_merged_dataset = merged_construction.copy()
final_merged_dataset['geometry_text'] = final_merged_dataset['geometry'].astype(str)
final_merged_dataset = final_merged_dataset.drop(columns=['geometry'])
final_merged_dataset.to_csv('final_merged_streets.csv', index=False)

# Data Visualization

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
